# Dataset Inspection: AIT-ADS

Reproduces the analysis + plots from the old `run_eda.py` / `run_eda_host.py` scripts for the AIT-ADS scenarios (fox, harrison, russellmitchell, santos, shaw, wardbeck, wheeler, wilson), via `thesis.data.eda` (computation) and `thesis.visualization.eda` (plotting). Output is written under `artifacts/experiments/run_eda/` the same way the script did, plus inline display here.

In [ ]:
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

REPO = Path.cwd().resolve().parents[2]  # src/thesis/notebooks -> repo root
sys.path.insert(0, str(REPO / 'src'))

import thesis.data.eda as de
from thesis.configs import load_scenarios
from thesis.visualization.eda import load_alerts, load_alert_groups, plot_label_distribution_table

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)

DATASET = 'ait-ads'
DATA_DIR = REPO / 'data' / 'alerts_csv'
ALERT_GROUPS_BASE_DIR = REPO / 'artifacts' / 'alert_groups'
scenarios = load_scenarios(DATASET)
print(f'Scenarios: {scenarios}')

run_ts = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
run_dir = REPO / 'artifacts' / 'experiments' / 'run_eda' / DATASET / f'run_{run_ts}_raw'
summary_dir = run_dir / 'summary'
print(f'Output dir: {run_dir}')

## Load alerts

In [ ]:
all_df = load_alerts(str(DATA_DIR), scenarios=scenarios, dataset=DATASET)
print(f'{len(all_df):,} alerts loaded.')
all_df.head()

## Phase 1: per-scenario analysis

Per-scenario stats, alert_group pair-frequency tables, and the per-scenario plots (signature event raster, short-descriptor event raster, alert_group size distribution, pair support scatter). Writes a text summary + CSVs per scenario, and builds the raw alert_group cache used by Phase 2's alert_group volume plots.

In [ ]:
for scenario in scenarios:
    scenario_df = all_df[all_df['scenario'] == scenario].copy()
    de.run_scenario_eda(
        scenario_df,
        scenario,
        out_path=run_dir / scenario,
        summary_path=summary_dir,
        alert_groups_base_dir=ALERT_GROUPS_BASE_DIR,
    )

## Cross-scenario tables

In [ ]:
overview_df = de.compute_overview_table(all_df)
overview_df.to_csv(run_dir / 'overview_table.csv', index=False)
overview_df

In [ ]:
label_dist_df = de.compute_label_distribution_table(all_df)
label_dist_df.to_csv(run_dir / 'label_distribution_table.csv', index=False)

fig, _ = plot_label_distribution_table(
    label_dist_df, out_path=str(run_dir / 'label_distribution_table.png')
)

## Phase 2: overview plots

Volume timeline, class balance, attack-type heatmap, top signatures, inter-arrival CDF, group sizes, scenario overview table, plus alert_group volume plots (built from the Phase 1 alert_group cache).

In [ ]:
groups_dir = de.groups_dir(ALERT_GROUPS_BASE_DIR)
groups_df = None
if groups_dir.exists():
    groups_df = load_alert_groups(str(groups_dir), scenarios=scenarios)
    print(f'{len(groups_df):,} alert_groups loaded from {groups_dir}.')
else:
    print(f'No alert_groups found at {groups_dir}; alert_group plots will be skipped.')

In [ ]:
plots = de.build_overview_plots(scenarios, all_df, dataset=DATASET, groups_df=groups_df)
de.save_overview_plots(plots, run_dir / 'plots', de.data_label(), close=False)

## Per-host analysis (AIT-ADS only)

Per-host alert-type vocabulary, burst/stream profile, and cross-host overlap -- answers whether per-host grouping keeps alert streams semantically pure. Reads from `artifacts/processed-data/<scenario>/alerts.json` (the tokenised/ingested alerts, not the raw CSV).

In [ ]:
host_run_dir = REPO / 'artifacts' / 'experiments' / 'run_eda_host' / f'run_{run_ts}'
for scenario in scenarios:
    de.run_host_scenario_eda(scenario, host_run_dir)